# Mcardiac perturbation

Change selected LR genes in source cardiomyocytes before simulation.

Keep one baseline run's simulation frames and set `baseline_frames` in `perturb/config.yaml`. These frames define future atrial and ventricular groups.


## Settings

Use `MAX_CELLS_PER_SAMPLE = None` to keep all cells. The example caps each sample at 1,000 cells.


In [1]:
from pathlib import Path
from datetime import datetime
import sys
import numpy as np

CWD = Path.cwd().resolve()
candidates = [CWD, CWD / "perturb", CWD / "experiments" / "Mcardiac" / "perturb"]
PERTURB_DIR = next(p for p in candidates if p.name == "perturb" and (p / "config.yaml").is_file())
PROJECT_ROOT = PERTURB_DIR.parents[2]
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from stvirtual.perturb import PerturbationSession, load_config, preflight

CONFIG_PATH = PERTURB_DIR / "config.yaml"
DEVICE = "cuda:0"
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
MAX_CELLS_PER_SAMPLE = 1000

In [2]:
audit = preflight(CONFIG_PATH)
print("Inputs ready:", audit["ready"], "Missing:", audit["missing_inputs"])
assert audit["ready"], audit
config, experiment, paths = load_config(CONFIG_PATH)
output = Path(paths["output"])
paths["output"] = str(output.with_name(f"{output.name}_{RUN_ID}"))
session = PerturbationSession.from_settings(config, experiment, paths, device=DEVICE)

Inputs ready: True Missing: []


## Apply perturbation


In [3]:
original = session.input()
query, report = session.apply_intervention(original)
display({key: value for key, value in report.items() if "obs_names" not in key})
print("Original cells:", original.n_obs, "Perturbed cells:", query.n_obs)
if MAX_CELLS_PER_SAMPLE is not None:
    sample_key = session.config["sample_key"]
    rng = np.random.default_rng(session.config["seed"])
    keep = []
    for sample in [session.config["source"], session.config["target"]]:
        rows = np.flatnonzero(query.obs[sample_key].astype(str).eq(sample))
        keep.extend(rng.choice(rows, min(len(rows), MAX_CELLS_PER_SAMPLE), replace=False))
    query = query[np.sort(keep)].copy()
print("Cells used:", query.obs[session.config["sample_key"]].value_counts().to_dict())

{'cell_state_transition_counts': {'other': 33139,
  'neither': 9114,
  'future_ventricular': 1245,
  'future_atrial': 1146},
 'down': 0.1,
 'up': 10.0,
 'missing_genes': {'ventricular': [], 'atrial': []},
 'intervention_samples': ['E9.5h'],
 'mode': 'cell_state_transition_lr'}

Original cells: 96315 Perturbed cells: 96315
Cells used: {'E9.5h': 1000, 'E11.5h': 1000}


## Update latent features

Reuse the trained scanVI model to save `X_scanvi_pert`.


In [4]:
query = session.reinfer(query, model=session.scanvi())
assert "X_scanvi_pert" in query.obsm
assert query.obsm["X_scanvi_pert"].shape[0] == query.n_obs
snapshot = session.save_input(query)
print(snapshot)


Seed set to 2025


INFO     File                                                                                                      
         /home/zhouyj/stVirtual_tutorial/experiments/Mcardiac/artifacts/rerun_default_scanvi_20261005_023545/checkp
         oints/scanvi/model.pt already downloaded                                                                  


Seed set to 2025


INFO     Found 100.0% reference vars in query data.                                                                


INFO     AnnData object appears to be a copy. Attempting to transfer setup.                                        


/home/zhouyj/stVirtual_tutorial/experiments/Mcardiac/artifacts/perturb/reciprocal_lr_seed2025_20261006_144426_073721/model_input.h5ad


## Run simulation


In [5]:
sample_key = session.config["sample_key"]
source = query[query.obs[sample_key].astype(str).eq(session.config["source"])].copy()
target = query[query.obs[sample_key].astype(str).eq(session.config["target"])].copy()
transport = session.transport(source, target)
ctx, pack, simulation = session.transition(query, transport)
print("Output directory:", session.output)

print("Simulation frames:", len(session.simulation["output_paths"]))

[ctx] scale 1/2: sparse matmul ...  (E=15000)
[ctx] scale 2/2: sparse matmul ...  (E=15000)


[ctx] scale 1/2: sparse matmul ...  (E=15000)
[ctx] scale 2/2: sparse matmul ...  (E=15000)


[diff] loaded /home/zhouyj/stVirtual_tutorial/experiments/Mcardiac/data/diff_map.csv Kmax= 3
[auto cap 3D] 2 {'src': {'q': 0.9, 'quantile_value': 2.0, 'mean': 1.1361502408981323, 'max': 3.0, 'n_cells_used': 852}, 'tgt': {'q': 0.9, 'quantile_value': 2.0, 'mean': 1.174545407295227, 'max': 4.0, 'n_cells_used': 825}, 'cap0': 2, 'capT': 2, 'cap': 2}


[diff] loaded /home/zhouyj/stVirtual_tutorial/experiments/Mcardiac/data/diff_map.csv Kmax= 3
[auto cap 3D] 2 {'src': {'q': 0.9, 'quantile_value': 2.0, 'mean': 1.1361502408981323, 'max': 3.0, 'n_cells_used': 852}, 'tgt': {'q': 0.9, 'quantile_value': 2.0, 'mean': 1.174545407295227, 'max': 4.0, 'n_cells_used': 825}, 'cap0': 2, 'capT': 2, 'cap': 2}


Output directory: /home/zhouyj/stVirtual_tutorial/experiments/Mcardiac/artifacts/perturb/reciprocal_lr_seed2025_20261006_144426_073721
Simulation frames: 21


## Save gene expression

After simulation, decode the saved frames from the repository root:

```bash
python experiments/decode_simulation.py --experiment experiments/Mcardiac --frame-dir <output-directory>/simulation
```
